In [ ]:
# [CELDA 0] SETUP — ejecutar UNA vez, luego Kernel -> Restart
# 1. Instala transformers==4.56.2 (version de entrenamiento, tiene DeepseekV2MoE)
# 2. Restaura modeling_deepseekocr2.py original desde HuggingFace (deshace parches previos)
# 3. Limpia cache HF
import subprocess, shutil, os

HF_TOKEN = "TU_HF_TOKEN"  # <-- pon tu token aqui

# 1. Instalar version correcta de transformers (sin unsloth, no se necesita para inferencia)
subprocess.run(["pip", "install",
    "transformers==4.56.2",
    "peft", "accelerate", "pillow",
    "huggingface_hub", "addict", "safetensors",
    "-q"], check=True)
print("transformers==4.56.2 instalado")

# 2. Re-descargar modeling_deepseekocr2.py original desde HF (revierte cualquier parche)
from huggingface_hub import hf_hub_download
original = hf_hub_download(
    "unsloth/DeepSeek-OCR-2",
    "modeling_deepseekocr2.py",
    token=HF_TOKEN,
    force_download=True,
)
dest = "./deepseek_ocr2/modeling_deepseekocr2.py"
if os.path.exists("./deepseek_ocr2"):
    shutil.copy(original, dest)
    print(f"modeling_deepseekocr2.py restaurado desde HF -> {dest}")
else:
    print("WARN: directorio ./deepseek_ocr2 no existe aun, se creara en celda 2")

# 3. Limpiar cache HF para regenerar desde el archivo restaurado
cache_path = os.path.expanduser("~/.cache/huggingface/modules/")
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("Cache HF limpiada")

print("=" * 50)
print("LISTO. Kernel -> Restart, luego ejecuta celdas 1, 2, 3")
print("=" * 50)

transformers==4.56.2 instalado


modeling_deepseekocr2.py: 0.00B [00:00, ?B/s]

modeling_deepseekocr2.py restaurado desde HF -> ./deepseek_ocr2/modeling_deepseekocr2.py
Cache HF limpiada
LISTO. Kernel -> Restart, luego ejecuta celdas 1, 2, 3


In [ ]:
# [CELDA 1] PASO 1 — Inspeccion de pesos del adaptador LoRA
# Verifica que no hay NaN, Inf ni capas muertas en adapter_model.safetensors
from safetensors import safe_open
from huggingface_hub import snapshot_download
import torch, os

HF_TOKEN = "TU_HF_TOKEN"  # <-- pon tu token aqui

if not os.path.exists("./adapter/adapter_model.safetensors"):
    snapshot_download("Lacax/deepseek_ocr_lora", local_dir="./adapter", token=HF_TOKEN)

results = {"critical": [], "lora_B_zero": [], "lora_A_dead": [], "other_warnings": []}
with safe_open("./adapter/adapter_model.safetensors", framework="pt") as f:
    for key in f.keys():
        tensor = f.get_tensor(key)
        if torch.isnan(tensor).any():
            results["critical"].append(f"NaN en {key}")
        if torch.isinf(tensor).any():
            results["critical"].append(f"Inf en {key}")
        if (tensor == 0).float().mean() > 0.9:
            (results["lora_B_zero"] if "lora_B" in key else results["lora_A_dead"]).append(key)
        if tensor.float().std() < 1e-7 and "lora_B" not in key:
            results["other_warnings"].append(f"Varianza nula: {key}")

print("CRITICOS:", results["critical"])
print("lora_B a cero (NORMAL):", len(results["lora_B_zero"]), "capas")
print("lora_A MUERTA (PROBLEMA):", results["lora_A_dead"])
print("OTRAS advertencias:", results["other_warnings"])

CRITICOS: []
lora_B a cero (NORMAL): 168 capas
lora_A MUERTA (PROBLEMA): []
OTRAS advertencias: []


In [ ]:
# [CELDA 2] PASO 2 — Carga del modelo base + adaptador LoRA
# Sin FastVisionModel: para inferencia basta con AutoModel + PeftModel.
# FastVisionModel.from_pretrained falla con deepseek_vl_v2 en unsloth >= 2025.x
import torch, os
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel
from huggingface_hub import snapshot_download

HF_TOKEN = "TU_HF_TOKEN"  # <-- pon tu token aqui

# Descargar modelo base (skip si ya existe)
if not os.path.exists("./deepseek_ocr2/config.json"):
    snapshot_download("unsloth/DeepSeek-OCR-2", local_dir="./deepseek_ocr2", token=HF_TOKEN)
else:
    print("deepseek_ocr2 ya descargado")

# Cargar modelo base con codigo custom (DeepseekOCR2)
model = AutoModel.from_pretrained(
    "./deepseek_ocr2",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)
tokenizer = AutoTokenizer.from_pretrained("./deepseek_ocr2", trust_remote_code=True)

# Cargar adaptador LoRA
model = PeftModel.from_pretrained(model, "Lacax/deepseek_ocr_lora", token=HF_TOKEN)
model.eval()
print(f"OK — dtype={model.dtype}, device={next(model.parameters()).device}")

deepseek_ocr2 ya descargado


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/691M [00:00<?, ?B/s]

OK — dtype=torch.bfloat16, device=cuda:0


In [ ]:
# [CELDA 3] TEST A — Sanidad basica con 5 tickets espanoles
# Requiere las 5 imagenes en el directorio de trabajo
# H1: max_new_tokens=4096, repetition_penalty=1.0, dynamic_preprocess siempre activo, preprocess_ticket

import torch
import math
import json
import os
import io
import cv2
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Any, Tuple
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
from deepseek_ocr2.modeling_deepseekocr2 import (
    format_messages,
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)

# ─── H1.5 — Preprocesado OpenCV: deskew + crop márgenes blancos ──────────────

def preprocess_ticket(img: Image.Image) -> Image.Image:
    """Deskew y crop de márgenes blancos para mejorar la calidad de entrada al modelo."""
    arr = np.array(img.convert("L"))
    _, thresh = cv2.threshold(arr, 200, 255, cv2.THRESH_BINARY_INV)
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) >= 50:
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = 90 + angle
        if abs(angle) > 0.5:
            h, w = arr.shape
            M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
            img = Image.fromarray(
                cv2.warpAffine(np.array(img), M, (w, h),
                               flags=cv2.INTER_CUBIC,
                               borderMode=cv2.BORDER_REPLICATE)
            )
    # Crop márgenes blancos
    arr2 = np.array(img.convert("L"))
    _, thresh2 = cv2.threshold(arr2, 200, 255, cv2.THRESH_BINARY_INV)
    rows = np.any(thresh2 > 0, axis=1)
    cols = np.any(thresh2 > 0, axis=0)
    if rows.any() and cols.any():
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]
        pad = 10
        img = img.crop((
            max(0, cmin - pad), max(0, rmin - pad),
            min(img.width, cmax + pad), min(img.height, rmax + pad)
        ))
    return img


# ─── DataCollator (igual que Celda G del notebook de entrenamiento) ───────────

@dataclass
class DeepSeekOCR2DataCollator:
    tokenizer: Any
    model: Any
    image_size: int = 768
    base_size: int = 1024
    crop_mode: bool = True
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(self, tokenizer, model, image_size=768, base_size=1024,
                 crop_mode=True, train_on_responses_only=True):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype
        self.train_on_responses_only = train_on_responses_only
        self.image_transform = BasicImageTransform(
            mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5), normalize=True
        )
        self.patch_size = 16
        self.downsample_ratio = 4
        if hasattr(tokenizer, 'bos_token_id') and tokenizer.bos_token_id is not None:
            self.bos_id = tokenizer.bos_token_id
        else:
            self.bos_id = 0

    def deserialize_image(self, image_data):
        if isinstance(image_data, Image.Image):
            img = image_data.convert("RGB")
        elif isinstance(image_data, dict) and 'bytes' in image_data:
            img = Image.open(io.BytesIO(image_data['bytes'])).convert("RGB")
        elif isinstance(image_data, str) and os.path.exists(image_data):
            img = Image.open(image_data).convert("RGB")
        else:
            raise ValueError(f"Unsupported image format: {type(image_data)}")
        return ImageOps.exif_transpose(img)

    def process_image(self, image):
        images_list, images_crop_list, images_spatial_crop = [], [], []
        if self.crop_mode:
            # H1.4: dynamic_preprocess activo para TODAS las resoluciones (no solo <=768px)
            images_crop_raw, crop_ratio = dynamic_preprocess(
                image, min_num=1, max_num=6,
                image_size=self.image_size, use_thumbnail=False
            )
            global_view = ImageOps.pad(
                image, (self.base_size, self.base_size),
                color=tuple(int(x * 255) for x in self.image_transform.mean)
            )
            images_list.append(self.image_transform(global_view).to(self.dtype))
            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])
            if width_crop_num > 1 or height_crop_num > 1:
                for crop_img in images_crop_raw:
                    images_crop_list.append(self.image_transform(crop_img).to(self.dtype))
            num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tokenized_image = ([self.image_token_id] * num_queries_base) * num_queries_base
            tokenized_image += [self.image_token_id]
        return images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio

    def process_single_sample(self, messages):
        images = []
        for message in messages:
            if "images" in message and message["images"]:
                for img_data in message["images"]:
                    if img_data is not None:
                        images.append(self.deserialize_image(img_data))
        if not images:
            raise ValueError("No images found in sample.")
        tokenized_str, images_seq_mask = [], []
        images_list, images_crop_list, images_spatial_crop = [], [], []
        prompt_token_count = -1
        assistant_started = False
        image_idx = 0
        tokenized_str.append(self.bos_id)
        images_seq_mask.append(False)
        for message in messages:
            role = message["role"]
            content = message["content"]
            if role == "<|Assistant|>":
                if not assistant_started:
                    prompt_token_count = len(tokenized_str)
                    assistant_started = True
                # Solo añadir EOS si hay contenido real (training).
                # En inferencia el content es "" — no añadir EOS o el modelo reinicia con BOS.
                if content.strip():
                    content = f"{content.strip()} {self.tokenizer.eos_token}"
            text_splits = content.split('<image>')
            for i, text_sep in enumerate(text_splits):
                tokenized_sep = text_encode(self.tokenizer, text_sep, bos=False, eos=False)
                tokenized_str.extend(tokenized_sep)
                images_seq_mask.extend([False] * len(tokenized_sep))
                if i < len(text_splits) - 1:
                    if image_idx >= len(images):
                        raise ValueError("Data mismatch: more <image> tokens than images.")
                    image = images[image_idx]
                    img_list, crop_list, spatial_crop, tok_img, _ = self.process_image(image)
                    images_list.extend(img_list)
                    images_crop_list.extend(crop_list)
                    images_spatial_crop.extend(spatial_crop)
                    tokenized_str.extend(tok_img)
                    images_seq_mask.extend([True] * len(tok_img))
                    image_idx += 1
        if not assistant_started:
            prompt_token_count = len(tokenized_str)
        images_ori = torch.stack(images_list, dim=0)
        images_spatial_crop_tensor = torch.tensor(images_spatial_crop, dtype=torch.long)
        if images_crop_list:
            images_crop = torch.stack(images_crop_list, dim=0)
        else:
            images_crop = torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype)
        return {
            "input_ids": torch.tensor(tokenized_str, dtype=torch.long),
            "images_seq_mask": torch.tensor(images_seq_mask, dtype=torch.bool),
            "images_ori": images_ori,
            "images_crop": images_crop,
            "images_spatial_crop": images_spatial_crop_tensor,
            "prompt_token_count": prompt_token_count,
        }

    def __call__(self, features):
        batch_data = []
        for feature in features:
            try:
                batch_data.append(self.process_single_sample(feature['messages']))
            except Exception as e:
                print(f"Error procesando muestra: {e}")
        if not batch_data:
            raise ValueError("No valid samples in batch")
        input_ids_list = [item['input_ids'] for item in batch_data]
        images_seq_mask_list = [item['images_seq_mask'] for item in batch_data]
        prompt_token_counts = [item['prompt_token_count'] for item in batch_data]
        input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence(images_seq_mask_list, batch_first=True, padding_value=False)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        labels[images_seq_mask] = -100
        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()
        images_batch = [(item['images_crop'], item['images_ori']) for item in batch_data]
        images_spatial_crop = torch.cat([item['images_spatial_crop'] for item in batch_data], dim=0)
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }


# ─── Test A ───────────────────────────────────────────────────────────────────

PROMPT = """<image>
Extract the following information from the receipt and return it STRICTLY as a valid JSON object matching this structure:

{
  "comercio": "string",
  "cif": "string",
  "fecha": "string",
  "total": "number",
  "items": [{"cantidad": "int", "descripcion": "string", "precio": "number"}]
}

NO other text. ONLY valid JSON.
"""

IMAGENES = [
    "recibo_almeria_079.jpg",
    "recibo_almeria_110.jpg",
    "recibo_almeria_111.jpg",
    "recibo_almeria_112.jpg",
    "recibo_almeria_114.jpg",
]

model.eval()

inference_collator = DeepSeekOCR2DataCollator(
    tokenizer=tokenizer,
    model=model,
    image_size=768,
    base_size=1024,
    crop_mode=True,
    train_on_responses_only=False,
)

resultados = []

for img_path in IMAGENES:
    print(f"\n{'='*60}")
    print(f"Imagen: {img_path}")
    # H1.5: preprocesado antes de inferencia
    image = preprocess_ticket(Image.open(img_path).convert("RGB"))

    sample = {
        "messages": [
            {"role": "<|User|>", "content": PROMPT, "images": [image]},
            {"role": "<|Assistant|>", "content": ""},
        ]
    }

    batch = inference_collator([sample])

    input_ids           = batch["input_ids"].to(model.device)
    attention_mask      = batch["attention_mask"].to(model.device)
    images              = [(c.to(model.device, model.dtype), o.to(model.device, model.dtype)) for c, o in batch["images"]]
    images_seq_mask     = batch["images_seq_mask"].to(model.device)
    images_spatial_crop = batch["images_spatial_crop"].to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids           = input_ids,
            attention_mask      = attention_mask,
            images              = images,
            images_seq_mask     = images_seq_mask,
            images_spatial_crop = images_spatial_crop,
            max_new_tokens      = 4096,   # H1.1: 1024 → 4096
            do_sample           = False,
            repetition_penalty  = 1.0,    # H1.2: 1.3 → 1.0 (desactivado)
        )

    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Tokens generados: {len(outputs[0]) - input_ids.shape[1]}")
    print("Resultado OCR:")
    print(response)

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print("ULTIMOS 600 CHARS DEL OUTPUT COMPLETO:")
    print(full_output[-600:])

    try:
        parsed = json.loads(response)
        print("-> JSON VALIDO")
        resultados.append({"imagen": img_path, "status": "PASS", "data": parsed})
    except json.JSONDecodeError as e:
        print(f"-> JSON INVALIDO: {e}")
        resultados.append({"imagen": img_path, "status": "FAIL", "raw": response})

print(f"\n{'='*60}")
print("RESUMEN TEST A:")
for r in resultados:
    print(f"  {r['imagen']}: {r['status']}")


In [ ]:
# [CELDA 4] Tests B, C, D, E — Protocolo PLAN_V4.md
# H1: max_new_tokens=4096, repetition_penalty=1.0, dynamic_preprocess siempre activo, preprocess_ticket
#
# Imagenes necesarias en el directorio de trabajo:
#   - Test B: test_b_noticket.jpg    (foto de paisaje/retrato, SIN texto de ticket)
#   - Test C: recibo_almeria_079.jpg (ya disponible del Test A)
#   - Test D: recibo_almeria_079.jpg (se recorta automaticamente por script)
#   - Test E: test_e_nuevo_comercio.jpg (ticket de comercio NO visto en entrenamiento)

import torch
import json
import math
import os
import io
import cv2
import numpy as np
from dataclasses import dataclass
from typing import Any
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
from unsloth import FastVisionModel
from deepseek_ocr2.modeling_deepseekocr2 import (
    format_messages,
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)

# ─── H1.5 — Preprocesado OpenCV ───────────────────────────────────────────────

def preprocess_ticket(img: Image.Image) -> Image.Image:
    """Deskew y crop de márgenes blancos para mejorar la calidad de entrada al modelo."""
    arr = np.array(img.convert("L"))
    _, thresh = cv2.threshold(arr, 200, 255, cv2.THRESH_BINARY_INV)
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) >= 50:
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = 90 + angle
        if abs(angle) > 0.5:
            h, w = arr.shape
            M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
            img = Image.fromarray(
                cv2.warpAffine(np.array(img), M, (w, h),
                               flags=cv2.INTER_CUBIC,
                               borderMode=cv2.BORDER_REPLICATE)
            )
    arr2 = np.array(img.convert("L"))
    _, thresh2 = cv2.threshold(arr2, 200, 255, cv2.THRESH_BINARY_INV)
    rows = np.any(thresh2 > 0, axis=1)
    cols = np.any(thresh2 > 0, axis=0)
    if rows.any() and cols.any():
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]
        pad = 10
        img = img.crop((
            max(0, cmin - pad), max(0, rmin - pad),
            min(img.width, cmax + pad), min(img.height, rmax + pad)
        ))
    return img


# ─── DataCollator ─────────────────────────────────────────────────────────────

@dataclass
class DeepSeekOCR2DataCollator:
    tokenizer: Any
    model: Any
    image_size: int = 768
    base_size: int = 1024
    crop_mode: bool = True
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(self, tokenizer, model, image_size=768, base_size=1024,
                 crop_mode=True, train_on_responses_only=True):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype
        self.train_on_responses_only = train_on_responses_only
        self.image_transform = BasicImageTransform(
            mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5), normalize=True
        )
        self.patch_size = 16
        self.downsample_ratio = 4
        if hasattr(tokenizer, 'bos_token_id') and tokenizer.bos_token_id is not None:
            self.bos_id = tokenizer.bos_token_id
        else:
            self.bos_id = 0

    def deserialize_image(self, image_data):
        if isinstance(image_data, Image.Image):
            img = image_data.convert("RGB")
        elif isinstance(image_data, dict) and 'bytes' in image_data:
            img = Image.open(io.BytesIO(image_data['bytes'])).convert("RGB")
        elif isinstance(image_data, str) and os.path.exists(image_data):
            img = Image.open(image_data).convert("RGB")
        else:
            raise ValueError(f"Unsupported image format: {type(image_data)}")
        return ImageOps.exif_transpose(img)

    def process_image(self, image):
        images_list, images_crop_list, images_spatial_crop = [], [], []
        if self.crop_mode:
            # H1.4: dynamic_preprocess activo para TODAS las resoluciones (no solo <=768px)
            images_crop_raw, crop_ratio = dynamic_preprocess(
                image, min_num=1, max_num=6,
                image_size=self.image_size, use_thumbnail=False
            )
            global_view = ImageOps.pad(
                image, (self.base_size, self.base_size),
                color=tuple(int(x * 255) for x in self.image_transform.mean)
            )
            images_list.append(self.image_transform(global_view).to(self.dtype))
            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])
            if width_crop_num > 1 or height_crop_num > 1:
                for crop_img in images_crop_raw:
                    images_crop_list.append(self.image_transform(crop_img).to(self.dtype))
            num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tokenized_image = ([self.image_token_id] * num_queries_base) * num_queries_base
            tokenized_image += [self.image_token_id]
        return images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio

    def process_single_sample(self, messages):
        images = []
        for message in messages:
            if "images" in message and message["images"]:
                for img_data in message["images"]:
                    if img_data is not None:
                        images.append(self.deserialize_image(img_data))
        if not images:
            raise ValueError("No images found in sample.")
        tokenized_str, images_seq_mask = [], []
        images_list, images_crop_list, images_spatial_crop = [], [], []
        prompt_token_count = -1
        assistant_started = False
        image_idx = 0
        tokenized_str.append(self.bos_id)
        images_seq_mask.append(False)
        for message in messages:
            role = message["role"]
            content = message["content"]
            if role == "<|Assistant|>":
                if not assistant_started:
                    prompt_token_count = len(tokenized_str)
                    assistant_started = True
                # Solo añadir EOS si hay contenido real (training).
                # En inferencia el content es "" — no añadir EOS o el modelo reinicia con BOS.
                if content.strip():
                    content = f"{content.strip()} {self.tokenizer.eos_token}"
            text_splits = content.split('<image>')
            for i, text_sep in enumerate(text_splits):
                tokenized_sep = text_encode(self.tokenizer, text_sep, bos=False, eos=False)
                tokenized_str.extend(tokenized_sep)
                images_seq_mask.extend([False] * len(tokenized_sep))
                if i < len(text_splits) - 1:
                    if image_idx >= len(images):
                        raise ValueError("Data mismatch: more <image> tokens than images.")
                    image = images[image_idx]
                    img_list, crop_list, spatial_crop, tok_img, _ = self.process_image(image)
                    images_list.extend(img_list)
                    images_crop_list.extend(crop_list)
                    images_spatial_crop.extend(spatial_crop)
                    tokenized_str.extend(tok_img)
                    images_seq_mask.extend([True] * len(tok_img))
                    image_idx += 1
        if not assistant_started:
            prompt_token_count = len(tokenized_str)
        images_ori = torch.stack(images_list, dim=0)
        images_spatial_crop_tensor = torch.tensor(images_spatial_crop, dtype=torch.long)
        if images_crop_list:
            images_crop = torch.stack(images_crop_list, dim=0)
        else:
            images_crop = torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype)
        return {
            "input_ids": torch.tensor(tokenized_str, dtype=torch.long),
            "images_seq_mask": torch.tensor(images_seq_mask, dtype=torch.bool),
            "images_ori": images_ori,
            "images_crop": images_crop,
            "images_spatial_crop": images_spatial_crop_tensor,
            "prompt_token_count": prompt_token_count,
        }

    def __call__(self, features):
        batch_data = []
        for feature in features:
            try:
                batch_data.append(self.process_single_sample(feature['messages']))
            except Exception as e:
                print(f"Error procesando muestra: {e}")
        if not batch_data:
            raise ValueError("No valid samples in batch")
        input_ids_list = [item['input_ids'] for item in batch_data]
        images_seq_mask_list = [item['images_seq_mask'] for item in batch_data]
        prompt_token_counts = [item['prompt_token_count'] for item in batch_data]
        input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence(images_seq_mask_list, batch_first=True, padding_value=False)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        labels[images_seq_mask] = -100
        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()
        images_batch = [(item['images_crop'], item['images_ori']) for item in batch_data]
        images_spatial_crop = torch.cat([item['images_spatial_crop'] for item in batch_data], dim=0)
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }


# ─── Prompt estándar (igual que Test A) ───────────────────────────────────────

PROMPT = """<image>
Extract the following information from the receipt and return it STRICTLY as a valid JSON object matching this structure:

{
  "comercio": "string",
  "cif": "string",
  "fecha": "string",
  "total": "number",
  "items": [{"cantidad": "int", "descripcion": "string", "precio": "number"}]
}

NO other text. ONLY valid JSON.
"""

# ─── Setup de inferencia ───────────────────────────────────────────────────────

FastVisionModel.for_inference(model)

inference_collator = DeepSeekOCR2DataCollator(
    tokenizer=tokenizer,
    model=model,
    image_size=768,
    base_size=1024,
    crop_mode=True,
    train_on_responses_only=False,
)


def run_inference(image: Image.Image) -> tuple[str, bool, dict | None]:
    """Ejecuta inferencia para una imagen. Devuelve (raw_text, json_valid, parsed_or_None)."""
    # H1.5: preprocesado antes de inferencia
    image = preprocess_ticket(image)
    sample = {
        "messages": [
            {"role": "<|User|>", "content": PROMPT, "images": [image]},
            {"role": "<|Assistant|>", "content": ""},
        ]
    }
    batch = inference_collator([sample])
    input_ids           = batch["input_ids"].to(model.device)
    attention_mask      = batch["attention_mask"].to(model.device)
    images              = [(c.to(model.device, model.dtype), o.to(model.device, model.dtype))
                           for c, o in batch["images"]]
    images_seq_mask     = batch["images_seq_mask"].to(model.device)
    images_spatial_crop = batch["images_spatial_crop"].to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids           = input_ids,
            attention_mask      = attention_mask,
            images              = images,
            images_seq_mask     = images_seq_mask,
            images_spatial_crop = images_spatial_crop,
            max_new_tokens      = 4096,   # H1.1: 1024 → 4096
            do_sample           = False,
            repetition_penalty  = 1.0,    # H1.2: 1.3 → 1.0 (desactivado)
        )

    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    try:
        parsed = json.loads(response)
        return response, True, parsed
    except json.JSONDecodeError:
        return response, False, None


# ─── TEST B — Imagen fuera de dominio (no-ticket) ─────────────────────────────

print("\n" + "="*60)
print("TEST B — Imagen fuera de dominio (no-ticket)")
print("="*60)

IMG_B = "vikings.jpg"

if not os.path.exists(IMG_B):
    print(f"SKIP: {IMG_B} no encontrada. Sube una foto de paisaje o retrato sin texto de ticket.")
else:
    image_b = Image.open(IMG_B).convert("RGB")
    raw_b, valid_b, parsed_b = run_inference(image_b)

    print(f"Tokens generados (aprox): ver output")
    print("Resultado OCR:")
    print(raw_b)

    if valid_b:
        campos_nulos = all(
            parsed_b.get(k) in (None, "", "null", "N/A", "unknown")
            for k in ["comercio", "cif", "total"]
        )
        print(f"\n-> JSON válido. Campos nulos/vacíos: {campos_nulos}")
        print(f"   comercio={parsed_b.get('comercio')}, total={parsed_b.get('total')}")
        if campos_nulos:
            print("-> TEST B: PASS (modelo devuelve campos vacíos/null para no-ticket)")
        else:
            print("-> TEST B: FAIL (modelo inventa datos para imagen sin ticket)")
    else:
        if any(kw in raw_b.lower() for kw in ["not a receipt", "no ticket", "no es", "no receipt", "cannot"]):
            print("-> TEST B: PASS (modelo indica explícitamente que no es un ticket)")
        else:
            print("-> TEST B: FAIL (JSON inválido y sin mensaje de rechazo claro)")
            print(f"   Raw: {raw_b[:300]}")


# ─── TEST C — Consistencia (misma imagen × 5) ─────────────────────────────────

print("\n" + "="*60)
print("TEST C — Consistencia (misma imagen x5)")
print("="*60)

IMG_C = "recibo_almeria_079.jpg"

if not os.path.exists(IMG_C):
    print(f"SKIP: {IMG_C} no encontrada.")
else:
    image_c = Image.open(IMG_C).convert("RGB")
    resultados_c = []

    for i in range(5):
        raw, valid, parsed = run_inference(image_c)
        resultados_c.append({"run": i+1, "valid": valid, "data": parsed, "raw": raw})
        status = "OK" if valid else "FAIL"
        comercio = parsed.get("comercio", "?") if parsed else "INVALID JSON"
        total    = parsed.get("total", "?")    if parsed else "INVALID JSON"
        print(f"  Run {i+1}: {status} | comercio={comercio} | total={total}")

    validos = [r for r in resultados_c if r["valid"]]
    if len(validos) == 5:
        totales   = set(str(r["data"].get("total"))   for r in validos)
        comercios = set(r["data"].get("comercio", "") for r in validos)
        consistent = len(totales) == 1 and len(comercios) == 1
        print(f"\n  Totales únicos: {totales}")
        print(f"  Comercios únicos: {comercios}")
        if consistent:
            print("-> TEST C: PASS (resultados idénticos en 5/5 ejecuciones)")
        else:
            print("-> TEST C: WARN (JSON válido en 5/5 pero con variaciones entre ejecuciones)")
    else:
        print(f"-> TEST C: FAIL ({5 - len(validos)}/5 ejecuciones devolvieron JSON inválido)")


# ─── TEST D — Campo faltante (ticket recortado programáticamente) ──────────────

print("\n" + "="*60)
print("TEST D — Ticket con campo faltante (recorte del 40% inferior)")
print("="*60)

IMG_D_BASE = "recibo_almeria_079.jpg"

if not os.path.exists(IMG_D_BASE):
    print(f"SKIP: {IMG_D_BASE} no encontrada.")
else:
    image_d_full = Image.open(IMG_D_BASE).convert("RGB")
    w, h = image_d_full.size
    image_d_cropped = image_d_full.crop((0, 0, w, int(h * 0.60)))
    image_d_cropped.save("test_d_cropped_preview.jpg")
    print(f"  Imagen recortada: {w}x{h} -> {w}x{int(h * 0.60)} (guardada en test_d_cropped_preview.jpg)")

    raw_d, valid_d, parsed_d = run_inference(image_d_cropped)
    print("Resultado OCR:")
    print(raw_d)

    if valid_d:
        total_val = parsed_d.get("total")
        total_ausente = total_val in (None, 0, "", "null", "N/A")
        print(f"\n  total extraído: {total_val}")
        if total_ausente:
            print("-> TEST D: PASS (modelo devuelve total null/vacío para imagen sin esa zona)")
        else:
            print(f"-> TEST D: WARN/FAIL (modelo extrajo total={total_val} de imagen sin zona de total)")
            print("   Verificar manualmente si el valor es inventado o si el total aparecía en otro lugar del ticket.")
    else:
        print(f"-> TEST D: FAIL (JSON inválido)")
        print(f"   Raw: {raw_d[:300]}")


# ─── TEST E — Comercio no visto en entrenamiento ──────────────────────────────

print("\n" + "="*60)
print("TEST E — Ticket de comercio no visto en entrenamiento")
print("="*60)

IMG_E = "recibo_diferente.jpg"

if not os.path.exists(IMG_E):
    print(f"SKIP: {IMG_E} no encontrada.")
    print("  Proporciona un ticket de CARREFOUR, LIDL, ALDI, EL CORTE INGLÉS u otro comercio")
    print("  que NO esté en el dataset de entrenamiento (no Mercadona, no DIA, no Alcarro).")
else:
    image_e = Image.open(IMG_E).convert("RGB")
    raw_e, valid_e, parsed_e = run_inference(image_e)

    print("Resultado OCR:")
    print(raw_e)

    COMERCIOS_ENTRENAMIENTO = {"mercadona", "grupo dia", "super alcarro", "dia"}

    if valid_e:
        comercio_extraido = (parsed_e.get("comercio") or "").lower().strip()
        overfitting = any(c in comercio_extraido for c in COMERCIOS_ENTRENAMIENTO)
        print(f"\n  comercio extraído: {parsed_e.get('comercio')}")
        if overfitting:
            print("-> TEST E: FAIL (overfitting — modelo devuelve comercio del dataset de entrenamiento)")
        else:
            print("-> TEST E: PASS (modelo extrae el comercio correcto o deja campo incompleto)")
    else:
        print(f"-> TEST E: FAIL (JSON inválido)")
        print(f"   Raw: {raw_e[:300]}")


# ─── RESUMEN FINAL ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("RESUMEN TESTS B-E")
print("="*60)
print("  Test A: PASS (5/5) — documentado en Documentacion/Inferencia_V4.md")
print("  Test B: ver resultado arriba")
print("  Test C: ver resultado arriba")
print("  Test D: ver resultado arriba")
print("  Test E: ver resultado arriba")
print()
print("Criterio de integración (PLAN_V4.md Paso 4):")
print("  A+C pasan + B+D parcialmente -> INTEGRAR en Scannet")
print("  A falla -> NO integrar, requiere V5")
print("  A pasa pero B o D fallan gravemente -> INTEGRAR CON CAUTELA + validación en /api/scan.ts")
